In [1]:
language = 'pt'

Importações


In [2]:
from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode
from gtts import  gTTS
import whisper
import openai
import os

In [3]:
!pip install git+https://github.com/openai/whisper.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
!pip install openai

In [5]:
!pip install gTTS

# 1. Gravação de Áudio Com Python🎤

In [6]:
# Referência: https://gist.github.com/korakot/c21c3476c024ad6d56d5f48b0bca92be

# Código JavaScript para gravar áudio do usuário usando a "MediaStream Recording API"
RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=5):
  # Executa o código JavaScript para gravar o áudio
  display(Javascript(RECORD))
  # Recebe o áudio gravado como resultado do JavaScript
  js_result = output.eval_js('record(%s)' % (sec * 1000))
   # Decodifica o áudio em base64
  audio = b64decode(js_result.split(',')[1])
  # Salva o áudio em um arquivo
  file_name = 'request_audio.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)
  # Retorna o caminho do arquivo de áudio (pasta padrão do Google Colab)
  return f'/content/{file_name}'

# Grava o áudio do usuário por um tempo determinado (padrão 5 segundos)
print('Ouvindo...\n')
record_file = record()

# Exibe o áudio gravado
display(Audio(record_file, autoplay=False))

Ouvindo...



<IPython.core.display.Javascript object>

# 2. Reconhecimento de Fala com Whisper (OpenAI) 🧠

In [7]:
# Selecione o modelo do Whisper que melhor atenda às suas necessidades:
# https://github.com/openai/whisper#available-models-and-languages
model = whisper.load_model("medium")

# Transcreve o audio gravado anteriormente.
result = model.transcribe(record_file, fp16=False, language=language)
transcription = result["text"]
print(transcription)

 Quais são os integrantes da banda Teimi Impala e qual o ano de formação?


# 3. Integração com a API do ChatGPT 💬

In [8]:


# Documentação Oficial da API OpenAI: https://platform.openai.com/docs/api-reference/introduction
# Informações sobre o Período Gratuito: https://help.openai.com/en/articles/4936830

# Substitua o texto "TODO" por sua API Key da OpenAI, ela será salva como uma variável de ambiente.
# a chave da API foi removida após a execução para proteção dos dados
os.environ['OPENAI_API_KEY'] = 'escondida'

In [9]:

# Configura a chave de API da OpenAI usando a variável de ambiente 'OPENAI_API_KEY'
client = openai.OpenAI(
    api_key=os.environ.get('OPENAI_API_KEY'),
)

# Envia uma requisição à API do ChatCompletion usando o modelo GPT-3.5 Turbo
# Lembrando que, a variável 'transcription' contém a transcrição do nosso áudio.
response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[ { "role": "user", "content": transcription } ]
)

# Obtém a resposta gerada pelo ChatGPT
chatgpt_response = response.choices[0].message.content
print(chatgpt_response)

Provavelmente você quis dizer Tame Impala, que é o projeto musical do músico Kevin Parker, formado em 2007 em Perth, Austrália.

Integrantes (principais ao longo do tempo):
- Kevin Parker – a figura central; compositor, multi-instrumentista, produtor. O "corpo" da banda é dele, com músicos de apoio nas gravações e ao vivo.
- Membros que já participaram com mais frequência (ao longo dos anos, como banda de apoio/colaboradores): Dominic Simper (guitarra/teclados), Nick Allbrook (baixo/vocais), Jay Watson (bateria/guitarra/teclados) e Cam Avery (baixo).

Observação: a formação exata muda conforme álbum, turnê e atuações ao vivo, já que Tame Impala é, em essência, o projeto solo de Parker com músicos de apoio.

Se quiser, posso montar uma linha do tempo com quem esteve envolvido em cada álbum ou turnê. E, se você realmente se referia a outra banda chamada “Teimi Impala”, me dê mais detalhes para eu confirmar.


# 4. Sintetizando a Resposta do ChatGPT Como Voz (gTTS) 🔊

In [10]:
# Cria um objeto gTTS com a resposta gerada pelo ChatGPT e a língua que será sintetizada em voz (variável "language").
gtts_object = gTTS(text=chatgpt_response, lang=language, slow=False)

# Salva o áudio da resposta no arquivo especificado (pasta padrão do Google Colab)
response_audio = "/content/response_audio.wav"
gtts_object.save(response_audio)

# Reproduz o áudio da resposta salvo no arquivo
display(Audio(response_audio, autoplay=True))